## 0. Setup (run every session)
Check the GPU type below matches what this job needs, then clone + install.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.exists('/content/COCO-Urdu'):
    !git clone -q -b v2 https://github.com/umair-hassan2/COCO-Urdu.git /content/COCO-Urdu
%cd /content/COCO-Urdu
!git pull -q
!pip install -q -e .

In [ ]:
import os
# Token source, in order: an HF_TOKEN already in the environment, else Colab Secrets
# (sidebar key icon -> add HF_TOKEN, enable for this notebook). Needs WRITE scope: every job
# pushes its outputs to the private artifacts repo via scripts/sync.py.
try:
    from google.colab import userdata
    os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN') or '')
except Exception:
    pass
assert os.environ.get('HF_TOKEN'), 'No HF_TOKEN. Add it in the Colab sidebar (key icon) as HF_TOKEN.'

# huggingface_hub reads HF_TOKEN from the environment, and the `!python` cells below
# are subprocesses that inherit it -- so no notebook_login() prompt is needed.
from huggingface_hub import whoami
_me = whoami(token=os.environ['HF_TOKEN'])
print('HF auth OK:', _me['name'], '| token role:', _me.get('auth', {}).get('accessToken', {}).get('role'))


## 1. Re-translate 318K captions with Gemma 3 27B — **A100 40 GB**, ~25–35 units
Fill in `MODEL` (a 4-bit AWQ/GPTQ `gemma-3-27b-it` repo that loads in vLLM) and `HF_REPO` (your private dataset for chunk checkpoints). No A100? set `MODEL='google/gemma-3-12b-it'`, `SYSTEM='gemma12b'` on an L4.

In [ ]:
!pip install -q 'vllm>=0.8.0'   # W4A16 compressed-tensors needs >= 0.8.0


In [ ]:
MODEL   = 'RedHatAI/gemma-3-27b-it-quantized.w4a16'   # compressed-tensors W4A16, ~19.7 GB, vLLM >= 0.8.0, fits A100 40 GB
SYSTEM  = 'gemma27b'
HF_REPO = 'umairhassan02/coco-urdu-v2-chunks'          # NEW private repo for resumable chunks; created automatically. NOT an input dataset.


### 1a. Sanity pass on 1,000 captions (~1 unit). Read the report before continuing.

In [ ]:
!python scripts/03_translate_gemma_vllm.py --system {SYSTEM} --model {MODEL} --hf-repo {HF_REPO} --limit 1000 --chunk-size 1000
!python scripts/03_translate_gemma_vllm.py --system {SYSTEM} --hf-repo {HF_REPO} --merge
!python scripts/04_sanity_check.py --translations data/translations/{SYSTEM}.parquet
print(open(f'results/sanity_{SYSTEM}.md').read())

### 1b. Full run. Re-run this cell after any disconnect; finished chunks are skipped.

In [ ]:
!python scripts/03_translate_gemma_vllm.py --system {SYSTEM} --model {MODEL} --hf-repo {HF_REPO} --chunk-size 20000
!python scripts/03_translate_gemma_vllm.py --system {SYSTEM} --hf-repo {HF_REPO} --merge
!python scripts/04_sanity_check.py --translations data/translations/{SYSTEM}.parquet

### 1c. Publish the outputs
Pushes to the private artifacts repo; the Mac picks them up with the same command (`pull`). No browser download.

In [ ]:
!python scripts/sync.py push data/translations/{SYSTEM}.parquet results/sanity_{SYSTEM}.md
